In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. 파일 경로
cafe_path = "data/소상공인시장진흥공단_상가(상권)정보_서울_202412.csv"
resident_path = "data/서울시 상권분석서비스(상주인구-상권).csv"
worker_path = "data/서울시 상권분석서비스(직장인구-상권).csv"
flow_path = "data/서울시 상권분석서비스(길단위인구-상권).csv"
shp_path = "data/서울시 상권분석서비스(영역-상권).shp"

# 2. 데이터 불러오기
df_cafe = pd.read_csv(cafe_path, encoding='utf-8')
df_resident = pd.read_csv(resident_path, encoding='cp949')
df_worker = pd.read_csv(worker_path, encoding='cp949')
df_flow = pd.read_csv(flow_path, encoding='cp949')
gdf_market = gpd.read_file(shp_path).to_crs("EPSG:4326")

# 3. 카페 데이터 → GeoDataFrame
df_cafe = df_cafe.dropna(subset=['경도', '위도'])
geometry = [Point(xy) for xy in zip(df_cafe['경도'], df_cafe['위도'])]
gdf_cafe = gpd.GeoDataFrame(df_cafe, geometry=geometry, crs="EPSG:4326")

# 4. 카페와 상권 Spatial Join (상권 내 포함 여부)
gdf_cafe_market = gpd.sjoin(gdf_cafe, gdf_market, how="left", predicate="within")
df_resident = df_resident.rename(columns={"상권_코드": "TRDAR_CD"})
df_worker = df_worker.rename(columns={"상권_코드": "TRDAR_CD"})
df_flow = df_flow.rename(columns={"상권_코드": "TRDAR_CD"})

df_resident['TRDAR_CD'] = df_resident['TRDAR_CD'].astype(str)
df_worker['TRDAR_CD'] = df_worker['TRDAR_CD'].astype(str)
df_flow['TRDAR_CD'] = df_flow['TRDAR_CD'].astype(str)
gdf_cafe_market['TRDAR_CD'] = gdf_cafe_market['TRDAR_CD'].astype(str)

# 5. 유동인구 정보 
df_resident = df_resident[["TRDAR_CD", "총_상주인구_수"]].drop_duplicates(subset="TRDAR_CD")
df_worker = df_worker[["TRDAR_CD", "총_직장_인구_수"]].drop_duplicates(subset="TRDAR_CD")
df_flow = df_flow[["TRDAR_CD", "총_유동인구_수"]].drop_duplicates(subset="TRDAR_CD")

df_merged = gdf_cafe_market.merge(df_resident[["TRDAR_CD", "총_상주인구_수"]], on="TRDAR_CD", how="left")
df_merged = df_merged.merge(df_worker[["TRDAR_CD", "총_직장_인구_수"]], on="TRDAR_CD", how="left")
df_merged = df_merged.merge(df_flow[["TRDAR_CD", "총_유동인구_수"]], on="TRDAR_CD", how="left")

# 6. 결과 확인용
result = df_merged[["상호명", "도로명주소", "TRDAR_CD", "TRDAR_CD_N", "총_상주인구_수", "총_직장_인구_수", "총_유동인구_수"]]
print(result.head())

result.to_csv("카페별_유동인구_매핑결과.csv", index=False, encoding="utf-8-sig")

          상호명                 도로명주소 TRDAR_CD                   TRDAR_CD_N  \
0       홈처치스쿨   서울특별시 동대문구 신이문로8길 5      nan                          NaN   
1    부동산임대김은숙   서울특별시 종로구 삼청로 122-1  3120005                    ì¼ì²­ë   
2   한국황토건축연구소  서울특별시 서초구 서초대로73길 40  3120189                    ê°ë¨ì­   
3  소망공인중개사사무소  서울특별시 성동구 용답중앙15길 18  3130072     ì±ëì©ëµìê°ìì¥   
4         고향집   서울특별시 종로구 돈화문로4길 30  3001494  ì¢ë¡Â·ì²­ê³ ê´ê´í¹êµ¬   

   총_상주인구_수  총_직장_인구_수   총_유동인구_수  
0       NaN        NaN        NaN  
1    1053.0     3091.0   319775.0  
2    6248.0    79639.0  7671468.0  
3    3965.0      505.0   873035.0  
4    1767.0    63286.0  8476380.0  


In [10]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import osmnx as ox
import networkx as nx
from sklearn.ensemble import RandomForestRegressor

# ------------------------- 파일 경로 -------------------------
cafe_path = "data/소상공인시장진흥공단_상가(상권)정보_서울_202412.csv"
resident_path = "data/서울시 상권분석서비스(상주인구-상권).csv"
worker_path = "data/서울시 상권분석서비스(직장인구-상권).csv"
flow_path = "data/서울시 상권분석서비스(길단위인구-상권).csv"
shp_path = "data/서울시 상권분석서비스(영역-상권).shp"

# ------------------------- 데이터 불러오기 -------------------------
df_cafe = pd.read_csv(cafe_path, encoding='utf-8')
df_resident = pd.read_csv(resident_path, encoding='cp949')
df_worker = pd.read_csv(worker_path, encoding='cp949')
df_flow = pd.read_csv(flow_path, encoding='cp949')
gdf_market = gpd.read_file(shp_path).to_crs("EPSG:4326")

# ------------------------- 카페 GeoDataFrame -------------------------
df_cafe = df_cafe.dropna(subset=['경도', '위도'])
gdf_cafe = gpd.GeoDataFrame(
    df_cafe,
    geometry=gpd.points_from_xy(df_cafe['경도'], df_cafe['위도']),
    crs="EPSG:4326"
)

# ------------------------- 상권 Spatial Join -------------------------
gdf_cafe = gpd.sjoin(gdf_cafe, gdf_market, how="left", predicate="within")
gdf_cafe["TRDAR_CD"] = gdf_cafe["TRDAR_CD"].astype(str)
for df in [df_resident, df_worker, df_flow]:
    df.rename(columns={"상권_코드": "TRDAR_CD"}, inplace=True)
    df["TRDAR_CD"] = df["TRDAR_CD"].astype(str)

# ------------------------- 상권별 인구 병합 -------------------------
df_resident = df_resident[["TRDAR_CD", "총_상주인구_수"]].drop_duplicates("TRDAR_CD")
df_worker = df_worker[["TRDAR_CD", "총_직장_인구_수"]].drop_duplicates("TRDAR_CD")
df_flow = df_flow[["TRDAR_CD", "총_유동인구_수", "X좌표", "Y좌표"]].drop_duplicates("TRDAR_CD")
gdf_cafe = gdf_cafe.merge(df_resident, on="TRDAR_CD", how="left")
gdf_cafe = gdf_cafe.merge(df_worker, on="TRDAR_CD", how="left")
gdf_cafe = gdf_cafe.merge(df_flow[["TRDAR_CD", "총_유동인구_수"]], on="TRDAR_CD", how="left")

# ------------------------- 반경 300m 가중 유동인구 -------------------------
gdf_flow_point = gpd.GeoDataFrame(
    df_flow,
    geometry=gpd.points_from_xy(df_flow["X좌표"], df_flow["Y좌표"]),
    crs="EPSG:4326"
).to_crs(epsg=5181)

cafe_buffer = gdf_cafe.to_crs(epsg=5181).copy()
cafe_buffer["buffer"] = cafe_buffer.geometry.buffer(300)
cafe_buffer = cafe_buffer.set_geometry("buffer")

joined = gpd.sjoin(cafe_buffer, gdf_flow_point.to_crs(epsg=5181), how="left", predicate="intersects")
joined["거리"] = joined.geometry.centroid.distance(joined["geometry_right"])
joined["가중치"] = 1 / (joined["거리"] + 1) ** 2

weighted = joined.groupby("상가업소번호").apply(
    lambda x: (x["총_유동인구_수"] * x["가중치"]).sum() / x["가중치"].sum()
).reset_index(name="버퍼_가중_유동인구")

gdf_cafe = gdf_cafe.merge(weighted, on="상가업소번호", how="left")

# ------------------------- 밀집도 계산 -------------------------
cafe_density = gdf_cafe.groupby("TRDAR_CD").size().reset_index(name="상권_내_카페수")
gdf_cafe = gdf_cafe.merge(cafe_density, on="TRDAR_CD", how="left")
gdf_cafe["1점포_당_유동인구"] = gdf_cafe["총_유동인구_수"] / gdf_cafe["상권_내_카페수"]

# ------------------------- 도보 거리 (도로 접근성) -------------------------
G = ox.graph_from_place("Seoul, South Korea", network_type="walk")
gdf_cafe["nearest_node"] = gdf_cafe.geometry.apply(lambda p: ox.nearest_nodes(G, p.x, p.y))
center_node = ox.nearest_nodes(G, 126.9784, 37.5666)  # 서울시청
gdf_cafe["도보_중심거리"] = gdf_cafe["nearest_node"].apply(
    lambda n: nx.shortest_path_length(G, n, center_node, weight="length")
    if nx.has_path(G, n, center_node) else None
)

# ------------------------- 머신러닝 예측 모델 (층수 제외) -------------------------
df_model = gdf_cafe.dropna(subset=["버퍼_가중_유동인구", "도보_중심거리"])
X = df_model[["버퍼_가중_유동인구", "상권_내_카페수", "도보_중심거리"]]
y = df_model["총_유동인구_수"]

model = RandomForestRegressor(random_state=42)
model.fit(X, y)
df_model["예측_유동인구"] = model.predict(X)

# ------------------------- 결과 저장 -------------------------
df_model[[
    "상호명", "도로명주소", "TRDAR_CD", "총_유동인구_수",
    "버퍼_가중_유동인구", "상권_내_카페수", "1점포_당_유동인구",
    "도보_중심거리", "예측_유동인구"
]].to_csv("정밀_유동인구_추정결과.csv", index=False, encoding="utf-8-sig")

KeyError: "['X좌표', 'Y좌표'] not in index"

In [3]:
print(gdf_cafe_market.columns)

Index(['상가업소번호', '상호명', '지점명', '상권업종대분류코드', '상권업종대분류명', '상권업종중분류코드',
       '상권업종중분류명', '상권업종소분류코드', '상권업종소분류명', '표준산업분류코드', '표준산업분류명', '시도코드',
       '시도명', '시군구코드', '시군구명', '행정동코드', '행정동명', '법정동코드', '법정동명', '지번코드',
       '대지구분코드', '대지구분명', '지번본번지', '지번부번지', '지번주소', '도로명코드', '도로명', '건물본번지',
       '건물부번지', '건물관리번호', '건물명', '도로명주소', '구우편번호', '신우편번호', '동정보', '층정보',
       '호정보', '경도', '위도', 'geometry', 'index_right', 'TRDAR_SE_C',
       'TRDAR_SE_1', 'TRDAR_CD', 'TRDAR_CD_N', 'XCNTS_VALU', 'YDNTS_VALU',
       'SIGNGU_CD', 'SIGNGU_CD_', 'ADSTRD_CD', 'ADSTRD_CD_', 'RELM_AR'],
      dtype='object')
